# 04 ? Technical Source Readiness

## Objective
Confirm that the five source CSV files can be loaded into the relational layer without changing their content. This notebook performs technical checks only; it does not clean, recode, rename, filter, or export microdata.

## Checks
- Resolve file paths, encoding, delimiter, and exact headers from the refreshed source catalog.
- Read every field as text and count rows without converting identifiers or categories.
- Check header agreement, row widths, and the selected technical key for missing or repeated values.
- Write aggregate readiness metrics only after all files have been read. No record values are displayed or stored in the report.

## Decision
The source is treated as already processed for survey production. This step verifies technical readiness and preserves all source names and values; it does not assert that the source is free of substantive survey limitations.

In [1]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate pyproject.toml from the current directory.")
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from mode_choice.source_readiness import inspect_source_readiness, write_readiness_report

readiness_results = inspect_source_readiness(PROJECT_ROOT)
readiness_report = write_readiness_report(PROJECT_ROOT, readiness_results)
readiness = pd.DataFrame(readiness_results)
print(f"Source tables inspected: {len(readiness)}")
print(f"Readiness report: {readiness_report.relative_to(PROJECT_ROOT)}")
display(readiness)
assert len(readiness) == 5, "Expected exactly five source CSV tables."
assert readiness["readiness_status"].eq("ready").all(), "At least one table needs technical review."

Source tables inspected: 5
Readiness report: docs\data_dictionary\standardization_readiness.csv


,source_path,source_encoding,source_delimiter,source_column_count,source_row_count,header_matches_schema,row_width_mismatch_count,technical_key_field,technical_key_empty_count,technical_key_whitespace_only_count,technical_key_duplicate_count,all_source_values_read_as_text,readiness_status
0,data/sources/raw/Database/a. Modulo hogares.csv,cp1252,;,27,22755,True,0,key_hg,0,0,0,True,ready
1,data/sources/raw/Database/b. Modulo vehiculos.csv,cp1252,;,19,17392,True,0,cod_vh,0,0,0,True,ready
2,data/sources/raw/Database/c. Modulo personas.csv,cp1252,;,40,67556,True,0,key_persona,0,0,0,True,ready
3,data/sources/raw/Database/d. Modulo viajes.csv,cp1252,;,47,100174,True,0,key_viaje,0,0,0,True,ready
4,data/sources/raw/Database/e. Modulo etapas.csv,cp1252,;,38,110881,True,0,key_etapa,0,0,0,True,ready
